In [ ]:
pip install pandas requests beautifulsoup4 tqdm lxml

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm
import re

# ====================== 1. Сбор реальных данных с GitHub ======================
def collect_from_github(max_repos=25):
    print("Сбор реальных данных с GitHub...")
    data = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    queries = ["курсовая работа python", "курсовая работа django", "курсовая flask python", "course work python"]

    for query in queries:
        url = f"https://api.github.com/search/repositories?q={query}&sort=stars&order=desc"
        
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code != 200:
                continue
                
            repos = resp.json().get('items', [])[:max_repos]

            for repo in tqdm(repos, desc=f"GitHub: {query[:40]}"):
                try:
                    readme_url = f"https://raw.githubusercontent.com/{repo['full_name']}/main/README.md"
                    readme_resp = requests.get(readme_url, headers=headers, timeout=10)
                    readme = readme_resp.text if readme_resp.status_code == 200 else ""

                    title = repo['name'].replace('-', ' ').replace('_', ' ').title()

                    data.append({
                        'title': title,
                        'speciality': 'Программная инженерия',
                        'description': repo.get('description') or "Разработка программного продукта",
                        'technologies': repo.get('language', 'Python'),
                        'full_text': readme[:15000],
                        'source': 'GitHub'
                    })
                except:
                    continue
                time.sleep(0.7)
        except:
            continue
    
    return data


# ====================== 2. Генерация синтетических данных ======================
def generate_explanatory_note(title, technologies="Python, Django, PostgreSQL"):
    note = f"""
ПОЯСНИТЕЛЬНАЯ ЗАПИСКА
к курсовой работе студента

Тема: «{title}»

Студент: Иванов И.И.
Группа: ИТ-21-1
Специальность: Программная инженерия

Руководитель: доц. Петров П.П.

Москва — 2026

1. ВВЕДЕНИЕ

Актуальность темы. В условиях цифровизации современного общества разработка {title.lower()} является высокоактуальной задачей.

Цель работы: разработка и реализация {title.lower()} с использованием современных технологий программирования.

Задачи исследования:
• проанализировать существующие аналоги;
• разработать архитектуру приложения;
• реализовать основную функциональность;
• провести тестирование разработанного решения;
• сформулировать выводы.

2. ОБЗОР ПРЕДМЕТНОЙ ОБЛАСТИ

На сегодняшний день существует большое количество решений в данной сфере. Однако многие из них обладают существенными недостатками...

3. АРХИТЕКТУРА ПРИЛОЖЕНИЯ

Для реализации проекта выбран язык программирования Python. В качестве основного фреймворка использован {technologies.split(',')[0]}. 
Для хранения данных применяется база данных {technologies.split(',')[-1] if ',' in technologies else 'SQLite'}.

4. РЕАЛИЗАЦИЯ

На данном этапе была успешно реализована основная функциональность приложения...

5. ТЕСТИРОВАНИЕ

Было проведено модульное и интеграционное тестирование. Результаты тестирования показали высокую стабильность работы системы.

6. ЗАКЛЮЧЕНИЕ

В ходе выполнения курсовой работы цель была полностью достигнута. Разработанное приложение соответствует современным требованиям.

Список использованной литературы
1. Мартин Р. Чистый код. — СПб.: Питер, 2022.
2. Фаулер М. Рефакторинг. — М.: Вильямс, 2019.
3. Официальная документация Python.
"""

    return note.strip()


# ====================== ГЛАВНАЯ ФУНКЦИЯ ======================
if __name__ == "__main__":
    all_data = []

    # 1. Собираем с GitHub
    github_data = collect_from_github(max_repos=30)
    all_data.extend(github_data)

    print(f"\nСобрано с GitHub: {len(github_data)} записей")

    # 2. Генерируем синтетические данные
    print("Генерация синтетических пояснительных записок...")

    synthetic_topics = [
        "Разработка веб-приложения для онлайн-магазина",
        "Система рекомендаций фильмов с использованием машинного обучения",
        "Мобильное приложение трекер привычек",
        "Telegram-бот для заказа еды в ресторане",
        "Анализатор тональности отзывов клиентов",
        "Система управления задачами для небольшой команды",
        "Онлайн-платформа для тестирования знаний студентов",
        "Парсер вакансий с HH.ru",
        "Игра «Змейка» с использованием Pygame",
        "Веб-сервис бронирования билетов в кинотеатр"
    ]

    for title in tqdm(synthetic_topics, desc="Генерация"):
        full_text = generate_explanatory_note(title)
        
        all_data.append({
            'title': title,
            'speciality': 'Программная инженерия',
            'description': f"Разработка {title.lower()} с использованием современных веб-технологий.",
            'technologies': "Python, Django, PostgreSQL, JavaScript",
            'full_text': full_text,
            'source': 'Synthetic'
        })

    # Создаём финальный датасет
    df = pd.DataFrame(all_data)

    # Сохраняем
    df.to_csv('coursework_dataset_final.csv', index=False, encoding='utf-8-sig')
    df.to_json('coursework_dataset_final.json', orient='records', force_ascii=False, indent=2)

    print(f"\n✅ Финальный датасет готов!")
    print(f"Всего записей: {len(df)}")
    print(f"Реальных (GitHub): {len(df[df['source']=='GitHub'])}")
    print(f"Синтетических: {len(df[df['source']=='Synthetic'])}")

    print("\nПервые 5 записей:")
    print(df[['title', 'source']].head())

Сбор реальных данных с GitHub...


GitHub: курсовая flask python: 100%|██████████| 6/6 [00:09<00:00,  1.58s/it]



Собрано с GitHub: 66 записей
Генерация синтетических пояснительных записок...


Генерация: 100%|██████████| 10/10 [00:00<00:00, 105649.97it/s]


✅ Финальный датасет готов!
Всего записей: 76
Реальных (GitHub): 66
Синтетических: 10

Первые 5 записей:
                               title  source
0               Labyrinth Generating  GitHub
1  Fashion Supply Chain Optimization  GitHub
2                        Gui Sqlite3  GitHub
3       Pythonprojectfortelegrammbot  GitHub
4                            Kursach  GitHub
